# Source Layer EDA — `source` schema
End-to-end exploratory analysis of every table landed by the MongoDB and
Databricks loaders (`scripts/run_mongo_job.py`, `scripts/run_databricks_job.py`).

**Covers:** inventory, schema, nulls, duplicates/keys, numeric distributions +
boxplots, dates/times (trends, lags, weekday), categoricals, cross-table join
coverage, and a data-quality summary.

Run from the repo root (`uv run jupyter lab`) — cell 1 resolves paths and
credentials from `.env` itself.


In [ ]:
import os
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import psycopg
import seaborn as sns
from dotenv import load_dotenv

warnings.filterwarnings("ignore", message="pandas only supports SQLAlchemy connectable")

REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
load_dotenv(REPO_ROOT / ".env")
sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 20)
plt.rcParams["figure.figsize"] = (10, 5)

SCHEMA = os.getenv("POSTGRES_SCHEMA_SOURCE", "source")


def pg_conn():
    return psycopg.connect(
        host=os.getenv("POSTGRES_HOST", "localhost"),
        port=os.getenv("POSTGRES_PORT", "5432"),
        dbname=os.getenv("POSTGRES_DB", "datawarehouse"),
        user=os.getenv("POSTGRES_USER", "postgres"),
        password=os.getenv("POSTGRES_PASSWORD", ""),
        autocommit=True,
    )


def read_table(name):
    with pg_conn() as conn:
        return pd.read_sql_query(f'SELECT * FROM "{SCHEMA}"."{name}"', conn)


TABLES = ["PX_CAT_G1V2", "CUST_AZ12", "LOC_A101", "cust_info", "prd_info", "sales_details"]
frames = {t: read_table(t) for t in TABLES}
print("loaded:", {t: frames[t].shape for t in TABLES})


## 1. Inventory — row counts and widths
Sanity check: does every expected table exist, and at what scale?
`sales_details` dominates; `PX_CAT_G1V2` is a tiny 37-row reference table.


In [ ]:
inv = pd.DataFrame(
    [{"table": t, "rows": len(frames[t]), "columns": frames[t].shape[1]} for t in TABLES]
).sort_values("rows", ascending=True)
inv

fig, ax = plt.subplots()
ax.barh(inv["table"], inv["rows"])
ax.set_xscale("log")
ax.set_xlabel("rows (log scale)")
ax.set_title("Source tables by row count")
plt.tight_layout()


## 2. Schema — dtypes and memory
Watch for: everything landing as `TEXT` (dates/numbers stored as strings
upstream), and `TIMESTAMPTZ` watermark columns.


In [ ]:
for t in TABLES:
    print(f"--- {t} memory={frames[t].memory_usage(deep=True).sum() / 1024:.1f} KB")
    print(frames[t].dtypes.to_string())
    print()


## 3. Null analysis — counts, % and heatmap
Anything above 0% is a staging-cleanse candidate. `prd_cost` / `prd_end_dt`
and customer name fields are the usual suspects.


In [ ]:
null_pct = pd.DataFrame(
    {t: frames[t].isna().mean().mul(100).round(2) for t in TABLES}
).fillna(0)
print(null_pct[null_pct.sum(axis=1) > 0].sort_values(
    by=list(null_pct.columns), ascending=False
).to_string() or "no nulls")

fig, axes = plt.subplots(2, 3, figsize=(15, 8))
for ax, t in zip(axes.flat, TABLES):
    s = frames[t].isna().mean().mul(100)
    hit = s[s > 0].sort_values()
    if hit.empty:
        ax.text(0.5, 0.5, "no nulls", ha="center", va="center", transform=ax.transAxes)
    else:
        hit.plot.barh(ax=ax)
    ax.set_title(f"{t} null %")
    ax.set_xlabel("pct null")
plt.tight_layout()


## 4. Duplicates and key uniqueness
Full-row duplicates plus uniqueness of the `id` merge key the loaders
upsert on. Any duplicate `id` means the last write silently won.


In [ ]:
dupes = pd.DataFrame(
    [
        {
            "table": t,
            "full_row_dupes": int(frames[t].duplicated().sum()),
            "id_unique": int(frames[t]["id"].nunique()) if "id" in frames[t] else None,
            "id_dupes": (len(frames[t]) - int(frames[t]["id"].nunique())) if "id" in frames[t] else None,
        }
        for t in TABLES
    ]
)
dupes


## 5. Numerics — coercion, describe, boxplots, outliers
`s sls_*` and `prd_cost` arrive as TEXT and must be coerced; non-parseable
tokens become NaN and are counted below. Boxplots use raw scale so skew and
outliers stay visible; IQR bounds quantify them.


In [ ]:
sales = frames["sales_details"].copy()
for c in ["sls_sales", "sls_quantity", "sls_price"]:
    sales[c] = pd.to_numeric(sales[c], errors="coerce")
print("unparseable tokens:", sales[["sls_sales", "sls_quantity", "sls_price"]].isna().sum().to_dict())

prd = frames["prd_info"].copy()
prd["prd_cost"] = pd.to_numeric(prd["prd_cost"], errors="coerce")

cust = frames["CUST_AZ12"].copy()
cust["birthdate"] = pd.to_datetime(cust["BDATE"], format="%Y-%m-%d", errors="coerce")
cust["age"] = ((pd.Timestamp.today().normalize() - cust["birthdate"]).dt.days // 365)
print("bad birthdates:", int(cust["birthdate"].isna().sum()),
      "| future birthdates:", int((cust["birthdate"] > pd.Timestamp.today()).sum()))

sales[["sls_sales", "sls_quantity", "sls_price"]].describe().T


In [ ]:
def iqr_outliers(s):
    s = s.dropna()
    q1, q3 = s.quantile(0.25), s.quantile(0.75)
    iqr = q3 - q1
    lo, hi = q1 - 1.5 * iqr, q3 + 1.5 * iqr
    return int(((s < lo) | (s > hi)).sum()), (float(lo), float(hi))

for col, frame in [("sls_sales", sales), ("sls_quantity", sales), ("sls_price", sales),
                   ("prd_cost", prd), ("age", cust)]:
    n, (lo, hi) = iqr_outliers(frame[col])
    print(f"{col}: outliers={n} bounds=({lo:.1f}, {hi:.1f})")

fig, axes = plt.subplots(2, 3, figsize=(16, 9))
for ax, (col, frame) in zip(axes.flat, [("sls_sales", sales), ("sls_quantity", sales),
                                        ("sls_price", sales), ("prd_cost", prd), ("age", cust)]):
    sns.boxplot(x=frame[col].dropna(), ax=ax)
    ax.set_title(f"{col} boxplot")
axes.flat[-1].set_visible(False)
plt.tight_layout()


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
sales["sls_sales"].dropna().plot.hist(bins=50, logy=True, ax=axes[0])
axes[0].set_title("sls_sales histogram (log count)")
sales["sls_price"].dropna().plot.hist(bins=50, logy=True, ax=axes[1])
axes[1].set_title("sls_price histogram (log count)")
plt.tight_layout()

print("quantity value counts:")
print(sales["sls_quantity"].value_counts(dropna=False).head(10).to_string())


## 6. Dates and times — batches, trends, lags, weekday
`updated_at` shows the loader batch; `_loaded_at` shows when it landed.
Customer creation, product lifespans and the sales calendar (monthly trend,
ship lag, weekday mix) follow.


In [ ]:
for t in TABLES:
    print(t, "updated_at distinct:", frames[t]["updated_at"].nunique(),
          "| min:", frames[t]["updated_at"].min(), "| max:", frames[t]["updated_at"].max())

cust_info = frames["cust_info"].copy()
cust_info["cst_create_date"] = pd.to_datetime(cust_info["cst_create_date"], errors="coerce")
monthly_cust = cust_info["cst_create_date"].dt.to_period("M").astype(str).value_counts().sort_index()
monthly_cust.tail(12).plot.bar(figsize=(12, 4), title="New customers per month (last 12)")
plt.tight_layout()


In [ ]:
sdt = sales.copy()
for c in ["sls_order_dt", "sls_ship_dt", "sls_due_dt"]:
    sdt[c] = pd.to_datetime(sdt[c], format="%Y%m%d", errors="coerce")
print("unparseable dates:", sdt[["sls_order_dt", "sls_ship_dt", "sls_due_dt"]].isna().sum().to_dict())

sdt["order_month"] = sdt["sls_order_dt"].dt.to_period("M").astype(str)
sdt["order_month"].value_counts().sort_index().plot(figsize=(14, 4), title="Orders per month")
plt.tight_layout()

sdt["ship_lag_days"] = (sdt["sls_ship_dt"] - sdt["sls_order_dt"]).dt.days
print("ship lag describe:")
print(sdt["ship_lag_days"].describe().to_string())
print("negative lags (shipped before ordered):", int((sdt["ship_lag_days"] < 0).sum()))

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
sdt["ship_lag_days"].dropna().plot.hist(bins=30, ax=axes[0])
axes[0].set_title("Ship lag days histogram")
sns.boxplot(x=sdt["ship_lag_days"].dropna(), ax=axes[1])
axes[1].set_title("Ship lag days boxplot")
plt.tight_layout()

sdt["weekday"] = sdt["sls_order_dt"].dt.day_name()
order = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]
sdt["weekday"].value_counts().reindex(order).plot.bar(figsize=(10, 4), title="Orders by weekday")
plt.tight_layout()


In [ ]:
pdt = prd.copy()
pdt["prd_start_dt"] = pd.to_datetime(pdt["prd_start_dt"], errors="coerce")
pdt["prd_end_dt"] = pd.to_datetime(pdt["prd_end_dt"], errors="coerce")
pdt["active_days"] = (pdt["prd_end_dt"] - pdt["prd_start_dt"]).dt.days
print("still active (no end date):", int(pdt["prd_end_dt"].isna().sum()), "/", len(pdt))
print(pdt["active_days"].describe().to_string())
pdt["active_days"].dropna().plot.hist(bins=30, figsize=(10, 4), title="Product lifespan days")
plt.tight_layout()


## 7. Categoricals — gender, country, category tree, lines
Top-N bars plus a CAT × SUBCAT cross-check against the 37-row reference.


In [ ]:
print("GEN:", cust["GEN"].value_counts(dropna=False).to_dict())
cust["GEN"].value_counts().plot.bar(title="Customers by gender")
plt.tight_layout()

loc = frames["LOC_A101"]
loc["CNTRY"].value_counts().head(15).sort_values().plot.barh(
    figsize=(8, 6), title="Top 15 countries")
plt.tight_layout()


In [ ]:
cat = frames["PX_CAT_G1V2"]
print(cat.groupby("CAT")["SUBCAT"].apply(list).to_string())
print()
print("MAINTENANCE:", cat["MAINTENANCE"].value_counts().to_dict())
pd.crosstab(cat["CAT"], cat["MAINTENANCE"])

print("cst_marital_status:", cust_info["cst_marital_status"].value_counts(dropna=False).to_dict())
print("cst_gndr:", cust_info["cst_gndr"].value_counts(dropna=False).to_dict())
print("prd_line:", prd["prd_line"].value_counts(dropna=False).to_dict())


## 8. Keys and join coverage — orphans kill marts
CID formats differ (`AW00028323` vs `AW-00025675`); normalize before
comparing. Coverage < 100% means orphan facts/dims downstream.


In [ ]:
def norm_cid(s):
    return s.astype(str).str.replace("-", "", regex=False).str.upper()

cust_ids = set(norm_cid(cust["CID"]))
loc_ids = set(norm_cid(loc["CID"]))
key_ids = set(cust_info["cst_key"].astype(str).str.upper())
print(f"CUST_AZ12={len(cust_ids)} LOC_A101={len(loc_ids)} cust_info keys={len(key_ids)}")
print("LOC not in CUST:", len(loc_ids - cust_ids), "| CUST not in cust_info:", len(cust_ids - key_ids),
      "| LOC not in cust_info:", len(loc_ids - key_ids))

sls_cust = set(sales["sls_cust_id"].astype(str))
cst_ids = set(cust_info["cst_id"].astype(str))
print(f"sales customers={len(sls_cust)} orphan vs cust_info: {len(sls_cust - cst_ids)} "
      f"({len(sls_cust - cst_ids) / max(len(sls_cust), 1) * 100:.1f}%)")

sls_prd = set(sales["sls_prd_key"].astype(str))
prd_keys = set(prd["prd_key"].astype(str))
print(f"sales products={len(sls_prd)} orphan vs prd_info: {len(sls_prd - prd_keys)} "
      f"({len(sls_prd - prd_keys) / max(len(sls_prd), 1) * 100:.1f}%)")

cov = pd.DataFrame([
    {"join": "LOC.CID -> CUST.CID", "coverage_pct": round(len(loc_ids & cust_ids) / max(len(loc_ids), 1) * 100, 2)},
    {"join": "CUST.CID -> cust_info.cst_key", "coverage_pct": round(len(cust_ids & key_ids) / max(len(cust_ids), 1) * 100, 2)},
    {"join": "sales.cust -> cust_info.cst_id", "coverage_pct": round(len(sls_cust & cst_ids) / max(len(sls_cust), 1) * 100, 2)},
    {"join": "sales.prd -> prd_info.prd_key", "coverage_pct": round(len(sls_prd & prd_keys) / max(len(sls_prd), 1) * 100, 2)},
])
cov
fig, ax = plt.subplots(figsize=(8, 3))
ax.barh(cov["join"], cov["coverage_pct"])
ax.set_xlim(0, 100)
ax.set_xlabel("coverage %")
ax.set_title("Join key coverage")
plt.tight_layout()


## 9. Data-quality summary
Read the tables above, then record verdicts here. Current known findings:

| # | Finding | Severity | Action for staging |
|---|---------|----------|--------------------|
| 1 | Dates/numbers stored as TEXT (`sls_*`, `prd_cost`, `BDATE`, `sls_*_dt`) | High | Coerce with `errors="coerce"`, quarantine rejects |
| 2 | Nulls in names / `prd_cost` / `prd_end_dt` | Medium | Decide null policy per column (see §3) |
| 3 | CID format mismatch (`AW…` vs `AW-…`) | High | Normalize (strip `-`, upper) before joins |
| 4 | Negative ship lags if any (see §6) | Medium | Flag impossible rows |
| 5 | Future birthdates if any (see §5) | Low | Flag or null out |
| 6 | Orphan keys (see §8) | High | Decide: drop facts vs keep with unknown dim |


In [ ]:
dq = pd.DataFrame([
    {"check": "null leaders", "value": null_pct.max().sort_values(ascending=False).head(3).to_dict()},
    {"check": "dup id rows", "value": int(sum(max(len(frames[t]) - int(frames[t]['id'].nunique()), 0) for t in TABLES if "id" in frames[t].columns))},
    {"check": "bad birthdates", "value": int(cust["birthdate"].isna().sum())},
    {"check": "negative ship lags", "value": int((sdt["ship_lag_days"] < 0).sum())},
    {"check": "orphan sales customers", "value": len(sls_cust - cst_ids)},
    {"check": "orphan sales products", "value": len(sls_prd - prd_keys)},
])
dq
